# Heavy American Option Pricing with Longstaff-Schwartz (LSM) and parfun

This notebook replaces the simple binomial tree iwth a **Longstaff-Schwartz Monte Carlo (LSM)** mode. LSM introduces:
- A full **stochastic process simulation** (GBM paths)
- **Cross-sectional regressions** at every exercise data
- Much higher computational cost

This is representative of **production American option engines** used for long-dated and complex payoffs.


## Program Structure

```mermaid
flowchart TB
    subgraph seq["Sequential"]
        direction TB
        S1["Option 1 20 vol scenarios"] --> S2["Option 2 20 vol scenarios"]
        S2 --> S3["Option 3 20 vol scenarios"]
        S3 --> S4["Option 4 20 vol scenarios"]
    end

    subgraph par["Parallel \u2014 4 workers"]
        direction TB
        subgraph W1["Worker 1"]
            A1["Option 1 
            20 vol scenarios"]
        end
        subgraph W2["Worker 2"]
            A2["Option 2
            20 vol scenarios"]
        end
        subgraph W3["Worker 3"]
            A3["Option 3
            20 vol scenarios"]
        end
        subgraph W4["Worker 4"]
            A4["Option 4
            20 vol scenarios"]
        end
        W1 & W2 & W3 & W4 --> C1(["concat results"])
    end
```

# Imports and Environment

In [1]:
import os
import sys
import time

import numpy as np

import parfun as pf

sys.stderr = open(os.devnull, "w")

/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: RankWarning: Polyfit may be poorly conditioned
/tmp/ipykernel_658766/1832001953.py:27: Ran


# This implementation follows the classical Longstaff-Schwartz (2001) algorithm.

Computational cost:
- Path simulation: O(paths x steps)
- Regression per step: O(paths x $basis^2$ x steps)

This quickly grows into **multi-minute workloads**.

In [2]:
def american_put_lsm(S, K, r, sigma, T, steps, paths, degree=6, seed=None):
    if seed is not None:
        np.random.seed(seed)

    dt = T / steps
    disc = np.exp(-r * dt)

    Z = np.random.normal(size=(paths, steps))
    S_paths = np.empty((paths, steps + 1))
    S_paths[:, 0] = S

    for t in range(steps):
        S_paths[:, t + 1] = S_paths[:, t] * np.exp(
            (r - 0.5 * sigma **2) * dt + sigma * np.sqrt(dt) * Z[:, t]
        )

    # Payoff at maturity
    cashflows = np.maximum(K - S_paths[:, -1], 0.0)

    # Backward inducion
    for t in range(steps - 1, 0, -1):
        itm = S_paths[:, t] < K
        X = S_paths[itm, t]
        Y = cashflows[itm] * disc

        if len(X) > degree:
            coeffs = np.polyfit(X, Y, degree)
            continuation = np.polyval(coeffs, X)
        else:
            continuation = np.zeros_like(X)

        exercise = K - X
        exercise_now = exercise > continuation

        cashflows[itm] = np.where(
            exercise_now,
            exercise,
            Y
        )

    return cashflows.mean() * disc


PATHS = 80_000  # Monte Carlo paths
STEPS = 252     # daily exercise dates
DEGREE = 6      # regression basis complexity


def price_with_scenarios(p, scenarios):
    S, K, r, _, T, St, N = p
    acc = [american_put_lsm(S, K, r, vol, T, St, N, DEGREE) for vol in scenarios]
    return sum(acc)

# Function to Parallelize

We price the same option under many volatility scenarios and repeat this across a large batch.
This mimics real-world **scenario analysis / stress testing** workloads.

In [3]:
def batch_price_with_scenarios(tasks, scenarios):
    return [price_with_scenarios(p, scenarios) / len(scenarios) for p in tasks]

# Parallelizing with ParFun

The only change needed is wrapping `batch_price_with_scenarios` with a `@pf.parfun` decorator.
The parallel version calls the **same function** — no logic is duplicated:

```diff
+ @pf.parfun(
+     split=pf.per_argument(tasks=pf.py_list.by_chunk),
+     combine_with=pf.py_list.concat,
+     fixed_partition_size=1,
+ )
  def batch_price_with_scenarios_w_parfun(tasks, scenarios):
+     return batch_price_with_scenarios(tasks, scenarios)
  ...
+ with pf.set_parallel_backend_context("scaler_local", n_workers=4):
+     result = batch_price_with_scenarios_w_parfun(BATCH, SCENARIOS)
```

In [4]:
@pf.parfun(
    split=pf.per_argument(tasks=pf.py_list.by_chunk),
    combine_with=pf.py_list.concat,
    fixed_partition_size=1,
)
def batch_price_with_scenarios_w_parfun(tasks, scenarios):
    return batch_price_with_scenarios(tasks, scenarios)

In [5]:
BATCH_SIZE = 4
SCENARIOS = np.linspace(0.15, 0.35, 20)

base_param = (100.0, 100.0, 0.05, 0.2, 1.0, STEPS, PATHS)
BATCH = [base_param] * BATCH_SIZE

start = time.time()
results_seq = batch_price_with_scenarios(BATCH, SCENARIOS)
seq_time = time.time() - start

start = time.time()
with pf.set_parallel_backend_context("scaler_local", n_workers=4):
    results_par = batch_price_with_scenarios_w_parfun(BATCH, SCENARIOS)
par_time = time.time() - start

print(f"Sequential: {seq_time / 60:.2f} minutes")
print(f"Parallel:   {par_time / 60:.2f} minutes")
print(f"Speedup:    {seq_time / par_time:.2f}x")

[INFO]2026-03-28 02:53:31+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 02:53:31+0800: ObjectStorageServer: start and listen to tcp://127.0.0.1:41473
[INFO]2026-03-28 02:53:31+0800: ObjectStorageServer: started
[INFO]2026-03-28 02:53:31+0800: logging to ('/dev/stdout',)
[INFO]2026-03-28 02:53:31+0800: use event loop: builtin
[INFO]2026-03-28 02:53:31+0800: ConfigController: scheduler_address = tcp://127.0.0.1:60969
[INFO]2026-03-28 02:53:31+0800: ConfigController: object_storage_address = tcp://127.0.0.1:41473
[INFO]2026-03-28 02:53:31+0800: ConfigController: monitor_address = None
[INFO]2026-03-28 02:53:31+0800: ConfigController: protected = True
[INFO]2026-03-28 02:53:31+0800: ConfigController: max_number_of_tasks_waiting = -1
[INFO]2026-03-28 02:53:31+0800: ConfigController: client_timeout_seconds = 60
[INFO]2026-03-28 02:53:31+0800: ConfigController: worker_timeout_seconds = 60
[INFO]2026-03-28 02:53:31+0800: ConfigController: object_retention_seconds = 60
[INFO]2026-03-28 02:

# Interpretation
- The sequential run should take **~10 minutes** on a single core (machine-dependent).
- The parfun version distributes work across available cores automatically.
- Speedup should approach the number of physical cores for this embarrasingly parallel workload.

This pattern closely matches real-world **risk, stress testing, and model validation** workloads.